# Limpieza y Transformación (ETL) del año 2020
Usando el dataframe ya en limpio del año 2019 decidi contraponer el año 2020 y 2021 para luego al final, cuando cree visualizaciones, poder tener un contexto de pre pandemia, pandemia en si y post pandemia para medir.

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

path_clean = "../data/clean/"
path_raw = "../data/raw/"

df_2019_clean = pd.read_csv(
    path_clean + "historico_2019_clean.csv",
    sep=",",
    encoding="utf-8-sig"
)

schema_2019 = df_2019_clean.columns.tolist()

print("Columnas schema 2019:", len(schema_2019))
schema_2019


Columnas schema 2019: 17


['periodo',
 'fecha',
 'desde',
 'hasta',
 'linea',
 'molinete',
 'estacion',
 'pax_pagos',
 'pax_pases_pagos',
 'pax_franq',
 'total',
 'hora_desde',
 'hora_hasta',
 'dia_semana',
 'mes',
 'dia_mes',
 'es_fin_semana']

# IMPORTANTE: DATA SCHEMA
Opte por usar el df limpio del 2019 como esquema inicial para respetar la estructura de datos presentada en aquel archivo. Tanto en este file como en el próximo a crear , año 2021, tendran que respetar las columnas de datos propuestas por el 2019 y en caso de tener inconsistencias, completar como lo amerite.


In [2]:
#Carga datos crudos

df_2020 = pd.read_csv(path_raw + "historico_2020.csv")

print(df_2020.shape)
df_2020.head(2)


(5781006, 10)


,FECHA,DESDE,HASTA,LINEA,MOLINETE,ESTACION,pax_pagos,pax_pases_pagos,pax_franq,pax_TOTAL
0,01/01/2020,08:00:00,08:15:00,LineaA,LineaA_Acoyte_N_Turn01,Acoyte,1.0,0.0,0.0,1.0
1,01/01/2020,08:00:00,08:15:00,LineaA,LineaA_Carabobo_E_Turn02,Carabobo,6.0,0.0,0.0,6.0


In [4]:
cols_2020 = set(df_2020.columns)
cols_2019 = set(schema_2019)

faltan_en_2020 = sorted(list(cols_2019 - cols_2020))
sobran_en_2020 = sorted(list(cols_2020 - cols_2019))

print("Faltan en 2020:", faltan_en_2020)
print("Sobran en 2020:", sobran_en_2020)


Faltan en 2020: ['dia_mes', 'dia_semana', 'es_fin_semana', 'hora_desde', 'hora_hasta', 'mes', 'periodo']
Sobran en 2020: []


### Alineación del esquema 2020 con el dataset de referencia 2019

Luego de estandarizar los nombres de columnas del dataset 2020, hice una
comparación contra el esquema de referencia definido a partir del dataset limpio
de 2019.

El análisis muestra que:
- No existen columnas adicionales en el dataset 2020 que no estén presentes en
  el esquema de referencia.
- Las únicas columnas ausentes corresponden a variables derivadas
  (`periodo`, `mes`, `dia_mes`, `dia_semana`, `es_fin_semana`,
  `hora_desde`, `hora_hasta`), las cuales no forman parte del dataset crudo y
  deben ser generadas durante la etapa de transformación.

Este resultado confirma la compatibilidad estructural entre los datasets y
valida el uso de un proceso ETL específico para el año 2020, orientado a la
creación de variables derivadas y a la alineación final del esquema, garantizando
consistencia y comparabilidad temporal con el período pre-pandemia.


In [5]:
df_2020.isna().sum()


fecha              1836
desde              1836
hasta              1836
linea              1836
molinete           1836
estacion           1836
pax_pagos          1836
pax_pases_pagos    1836
pax_franq          1836
total              1836
dtype: int64

### Auditoría de valores faltantes (2020)

Se detectó una proporción baja de valores faltantes (~0.032%) y, de forma consistente, el porcentaje es idéntico en todas las columnas.  
Esto sugiere la existencia de registros incompletos (filas con múltiples campos faltantes simultáneamente), más que un problema aislado de una variable específica.  
Se decide conservar estos registros y manejar los nulos mediante tipados que soporten valores faltantes y reglas robustas de transformación, evitando eliminar datos.


In [7]:
df_2020.loc[df_2020["fecha"].isna(), ["fecha","desde","hasta","linea","estacion"]].head(5)

,fecha,desde,hasta,linea,estacion
3917825,NaN,NaN,NaN,NaN,NaN
3917826,NaN,NaN,NaN,NaN,NaN
3917827,NaN,NaN,NaN,NaN,NaN
3917828,NaN,NaN,NaN,NaN,NaN
3917829,NaN,NaN,NaN,NaN,NaN


In [8]:
df_2020.loc[df_2020["fecha"].notna(), ["fecha","desde","hasta","linea","estacion"]].head(5)


,fecha,desde,hasta,linea,estacion
0,01/01/2020,08:00:00,08:15:00,LineaA,Acoyte
1,01/01/2020,08:00:00,08:15:00,LineaA,Carabobo
2,01/01/2020,08:00:00,08:15:00,LineaA,Castro Barros
3,01/01/2020,08:00:00,08:15:00,LineaA,Castro Barros
4,01/01/2020,08:00:00,08:15:00,LineaA,Congreso


In [9]:
# 1) eliminar filas donde TODAS las columnas son NaN
df_2020 = df_2020.dropna(how="all")


In [10]:
# 2) chequeo: nulos por columna
df_2020.isna().sum().head(10), df_2020.shape

(fecha              0
 desde              0
 hasta              0
 linea              0
 molinete           0
 estacion           0
 pax_pagos          0
 pax_pases_pagos    0
 pax_franq          0
 total              0
 dtype: int64,
 (5779170, 10))

In [12]:
rename_map = {
    "FECHA": "fecha",
    "DESDE": "desde",
    "HASTA": "hasta",
    "LINEA": "linea",
    "MOLINETE": "molinete",
    "ESTACION": "estacion",
    "pax_TOTAL": "total",
}

df_2020 = df_2020.rename(columns=rename_map)
df_2020.columns

Index(['fecha', 'desde', 'hasta', 'linea', 'molinete', 'estacion', 'pax_pagos', 'pax_pases_pagos', 'pax_franq', 'total'], dtype='object')

In [13]:
# strings
df_2020["desde"] = df_2020["desde"].astype("string").str.strip().replace("", pd.NA)
df_2020["hasta"] = df_2020["hasta"].astype("string").str.strip().replace("", pd.NA)

df_2020["linea"] = df_2020["linea"].astype("string").str.strip().str.upper()
df_2020["estacion"] = df_2020["estacion"].astype("string").str.strip().str.title()
df_2020["molinete"] = df_2020["molinete"].astype("string").str.strip().str.upper()

# horas desde string HH:MM:SS
df_2020["hora_desde"] = pd.to_numeric(
    df_2020["desde"].str.split(":").str[0],
    errors="coerce"
).astype("Int8")

df_2020["hora_hasta"] = pd.to_numeric(
    df_2020["hasta"].str.split(":").str[0],
    errors="coerce"
).astype("Int8")


In [14]:
df_2020["fecha"] = df_2020["fecha"].astype("string").str.strip()

df_2020["fecha"] = pd.to_datetime(
    df_2020["fecha"],
    format="%d/%m/%Y",
    errors="coerce"
)

df_2020["fecha"].isna().sum()


np.int64(1655090)